# 03 — Error Analysis and Word-Order Study

This notebook examines the most informative mistakes made by the best-performing model and compares them against the MLP baseline. The goal is to understand recurring failure modes qualitatively and to demonstrate, with concrete examples, why a mean-pooled MLP is limited when sentiment depends on word order or local contextual structure.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style='whitegrid')
exp_dir = Path('experiments')
table_dir = Path('report/tables')
table_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
bilstm_preds = pd.read_csv(exp_dir / 'bilstm_attn_main' / 'preds_test.csv')
mlp_preds = pd.read_csv(exp_dir / 'mlp_main' / 'preds_test.csv')

merged = bilstm_preds.merge(mlp_preds[['idx', 'pred', 'prob_pos']].rename(columns={'pred': 'mlp_pred', 'prob_pos': 'mlp_prob_pos'}), on='idx', how='inner')
merged['text_length'] = merged['text'].fillna('').str.split().str.len()
merged['bilstm_confidence'] = np.where(merged['pred'] == 1, merged['prob_pos'], 1 - merged['prob_pos'])
merged['mlp_confidence'] = np.where(merged['mlp_pred'] == 1, merged['mlp_prob_pos'], 1 - merged['mlp_prob_pos'])
merged.head()

In [ ]:
bilstm_errors = merged[merged['pred'] != merged['label']].copy()
bilstm_errors['confidence'] = bilstm_errors['bilstm_confidence']
bilstm_errors = bilstm_errors.sort_values(['confidence', 'text_length'], ascending=[True, False])
bilstm_errors.head()

## BiLSTM-Attn misclassifications

The cells below sample a small but diverse set of mistakes from the BiLSTM-Attn model. Each example is chosen to highlight a different failure mode such as negation, long context, mixed sentiment, or short/sparse input.

In [ ]:
negation_terms = ['not', 'no', 'never', 'without']
positive_seeds = ['great', 'love', 'excellent', 'amazing', 'wonderful', 'best']
negative_seeds = ['bad', 'worst', 'boring', 'terrible', 'awful', 'poor']

def has_any(text: str, words: list[str]) -> bool:
    lower = text.lower()
    return any(word in lower for word in words)

q1 = bilstm_errors['text_length'].quantile(0.25)
q3 = bilstm_errors['text_length'].quantile(0.75)

selected = []
used_idx = set()

for _, row in bilstm_errors[bilstm_errors['text'].apply(lambda t: has_any(t, negation_terms))].sort_values('confidence').head(2).iterrows():
    if row['idx'] not in used_idx:
        selected.append((row, 'Negation cue'))
        used_idx.add(row['idx'])

for _, row in bilstm_errors[bilstm_errors['text_length'] >= q3].sort_values('confidence').head(2).iterrows():
    if row['idx'] not in used_idx:
        selected.append((row, 'Long example'))
        used_idx.add(row['idx'])

for _, row in bilstm_errors[bilstm_errors['text'].apply(lambda t: has_any(t, positive_seeds) and has_any(t, negative_seeds))].sort_values('confidence').head(2).iterrows():
    if row['idx'] not in used_idx:
        selected.append((row, 'Mixed sentiment'))
        used_idx.add(row['idx'])

for _, row in bilstm_errors[bilstm_errors['text_length'] <= q1].sort_values('confidence').head(2).iterrows():
    if row['idx'] not in used_idx:
        selected.append((row, 'Short / sparse'))
        used_idx.add(row['idx'])

if len(selected) < 5:
    remaining = bilstm_errors[~bilstm_errors['idx'].isin(used_idx)].sort_values('confidence').head(8 - len(selected))
    for _, row in remaining.iterrows():
        selected.append((row, 'Additional error'))

selected_rows = []
for row, category in selected[:8]:
    selected_rows.append({
        'idx': int(row['idx']),
        'category': category,
        'true_label': int(row['label']),
        'pred_label': int(row['pred']),
        'confidence': float(row['confidence']),
        'text': row['text'][:400] + ('...' if len(row['text']) > 400 else ''),
    })

error_examples_df = pd.DataFrame(selected_rows)
display(error_examples_df[['idx', 'category', 'true_label', 'pred_label', 'confidence', 'text']])
(table_dir / 'error_examples.md').write_text(error_examples_df[['idx', 'category', 'true_label', 'pred_label', 'confidence', 'text']].to_markdown(index=False), encoding='utf-8')

### Failure categories

- Negation cue
- Long example
- Mixed sentiment
- Short / sparse
- Additional error

In [ ]:
disagree = merged[(merged['pred'] != merged['mlp_pred']) & (merged['pred'] == merged['label'])].copy()
disagree['mlp_correct'] = disagree['mlp_pred'] == disagree['label']
disagree = disagree.sort_values(['text_length', 'bilstm_confidence'], ascending=[False, False])

pattern_keywords = ['not', 'never', 'without', 'but', 'although', 'yet', 'only', 'however', 'despite', 'despite']
def order_hinge_score(text: str) -> int:
    lower = text.lower()
    return sum(1 for kw in pattern_keywords if kw in lower)

disagree['order_score'] = disagree['text'].apply(order_hinge_score)
disagree = disagree.sort_values(['order_score', 'text_length', 'bilstm_confidence'], ascending=[False, False, False])
disagree_selected = disagree.head(5).copy()
disagree_selected['text_preview'] = disagree_selected['text'].str.slice(0, 400) + disagree_selected['text'].str.len().map(lambda n: '...' if n > 400 else '')
disagree_selected[['idx', 'text_preview', 'label', 'mlp_pred', 'pred', 'mlp_prob_pos', 'prob_pos']]

## Why MLP is limited for word order

The cross-model examples show that the MLP baseline often struggles when sentiment depends on the arrangement of words rather than on the words themselves. In sentences with negation, such as constructions containing *not*, *never*, or *without*, the meaning of a positive cue can be reversed by a nearby negative operator. Similarly, in mixed-sentiment reviews, the overall label may depend on which clause comes first or on whether a contrastive phrase like *but* flips the interpretation of the earlier content.

This behavior is consistent with the MLP architecture: after embedding, it applies mean pooling across time, which compresses the entire sequence into an order-agnostic average. Once the token sequence has been pooled this way, information about local word order, negation scope, and clause structure is largely discarded. By contrast, the BiLSTM-Attn model preserves sequential context and can focus attention on the most sentiment-bearing positions, so it is better able to resolve cases where the same words appear but in a different syntactic arrangement. The examples therefore provide concrete evidence that word order is not just a minor detail for IMDb sentiment classification; it is often essential for the final decision.

In [ ]:
display(disagree_selected[['idx', 'label', 'mlp_pred', 'pred', 'mlp_prob_pos', 'prob_pos', 'text_preview']])
(table_dir / 'word_order_examples.md').write_text(disagree_selected[['idx', 'label', 'mlp_pred', 'pred', 'mlp_prob_pos', 'prob_pos', 'text_preview']].to_markdown(index=False), encoding='utf-8')